# Prepara y estructura datos con Python III 🐍🖥️✍️

## Objetivos academicos

- Realizar resúmenes agregados con datos
- Combinar datos entre sí
- Usar gráficos para resumir la información 


## Pandas para análisis de datos 🐼📊

---

### Resumen de las sesiones anteriores 🔃

Trabajaremos con dos datasets: **openaq_pm25**, que contiene niveles de contaminación atmosférica por país, e **IHME_GBD_2021**, con indicadores de salud global. Usaremos ambos para practicar el resumen de datos con pandas y la unión de tablas mediante claves comunes.


En sesiones anteriores hemos trabajado con pandas 🐼:

- Carga de datos
- Exploración de datos
- Limpieza de datos
    - Conversión de variables
    - Limpieza de nulos y duplicados
 




In [ ]:
import pandas as pd  # ¿Pa' qué servia esto?

#### *Carga los datos*

In [ ]:
pm25=pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/openaq_pm25.csv')


In [ ]:
gbd=pd.read_csv('https://practicum-content.s3.us-west-1.amazonaws.com/datasets/IHME_GBD_2021.csv')

In [ ]:
pm25.info()

In [ ]:
gbd.info()

#### Limpieza de  *pm25*

In [ ]:
#Estandarización de datos ¿Puedes describir que se hace en cada linea?
pm25['country_name'] = pm25['country_name'].str.title()
pm25['lastUpdated'] = pd.to_datetime(pm25['lastUpdated'],errors='coerce')
pm25['year_id'] = pm25['lastUpdated'].dt.year


In [ ]:
#Selección de columnas
pm25 = pm25[['parameter',	'unit',	'value', 'country_name', 'lastUpdated','year_id']]

In [ ]:
#Filtrado por condiciones
pm25 = pm25[pm25['country_name'].isin(['Argentina', 'Brazil', 'Costa Rica','Mexico','Chile'])]


In [ ]:
#Eliminando nulos y duplicados
pm25= pm25.dropna()
pm25 = pm25.drop_duplicates()



In [ ]:

pm25.info(),pm25.head()

#### Limpieza de  *gbd*

In [ ]:
#Renombrado de columnas
gbd = gbd.rename(columns={'location_name': 'country_name'})

In [ ]:
gbd['country_name'] = gbd['country_name'].str.title()

In [ ]:
gbd = gbd[['country_name', 'year_id', 'pathogen', 'metric', 'measure_name', 'age_group_name']]

In [ ]:
gbd = gbd[gbd['country_name'].isin(['Argentina', 'Brazil', 'Costa Rica','Mexico'])]

In [ ]:
gbd = gbd[gbd['measure_name'] == 'Deaths']


In [ ]:
gbd = gbd[gbd['pathogen'] == 'Influenza']


In [ ]:
gbd = gbd[gbd['age_group_name'] == 'All Ages']

In [ ]:
gbd = gbd[gbd['metric'] == 'Percent']


In [ ]:
gbd = gbd[gbd['year_id'] == 2021]


In [ ]:
gbd.info(),gbd.head()

### Resumir datos (`group by`+ `agg`)


Para resumir valores de una columna usamos sintaxis como `df["col"].mean()` o `df["col"].min()`, obteniendo métricas rápidas sin agrupar datos.


In [ ]:
df_ar = pm25[pm25["country_name"] == "Argentina"]
df_mx = pm25[pm25["country_name"] == "Mexico"]

In [ ]:
# Métricas Argentina
mean_ar = df_ar["value"].mean()
median_ar = df_ar["value"].median()
min_ar = df_ar["value"].min()
max_ar = df_ar["value"].max()
std_ar = df_ar["value"].std()

In [ ]:
print("Argentina:   Media:", mean_ar, "Mediana:",median_ar, " Min-Max:", min_ar,"-",max_ar, " Desv. est:", std_ar)

In [ ]:
# Métricas México
mean_mx = df_mx["value"].mean()
median_mx = df_mx["value"].median()
min_mx = df_mx["value"].min()
max_mx = df_mx["value"].max()
std_mx = df_mx["value"].std()

In [ ]:
print("México:   Media:", mean_mx, "Mediana:",median_mx, " Min-Max:", min_mx,"-",max_mx, " Desv. est:", std_mx)

El método `groupby()` en pandas permite agrupar datos por una columna y calcular métricas sobre cada grupo. Su sintaxis básica es: `df.groupby("col")["valor"].mean()`. Puedes usar funciones de agregación como `mean()`, `median()`, `min()`, `max()`, `std()`, `count()` o combinarlas con `.agg()` para calcular varias métricas a la vez: `df.groupby("col")["valor"].agg(["mean","min","max"])`. 

In [ ]:
pm25.groupby('country_name')['value'].mean()

El resultado de `groupby()` genera un índice basado en la columna agrupada, por eso usamos `reset_index()` para convertirlo nuevamente en una columna normal y obtener un DataFrame limpio y fácil de manipular.

In [ ]:
pm25.groupby('country_name')['value'].mean().reset_index()

In [ ]:
pm25.groupby('country_name')['value'].agg(["mean", "sum", "count"]).reset_index()

In [ ]:
group_cols = ['parameter', 'unit', 'country_name', 'year_id']
pm25_agg = pm25.groupby(group_cols, dropna=False)['value'].mean().reset_index()
pm25_agg

### Unir datos (`merge`)

El método `merge()` permite unir dos DataFrames usando una columna en común. Su sintaxis básica es: `df_a.merge(df_b, how="left", left_on="id", right_on="id")`, donde `how` define el tipo de unión y las claves indican qué columnas se usarán para combinar los datos.


In [ ]:
df_a=pd.DataFrame({'id':[1,2,3,4],'value':[15,15,87,78]})
df_b=pd.DataFrame({'id':[1,3,5],'tag':['a','b','c']})
df_a,df_b


En `merge()`, una unión **left** conserva todas las filas del DataFrame izquierdo y solo agrega coincidencias del derecho. En cambio, una unión **inner** solo devuelve las filas donde ambas tablas tienen coincidencias en la clave de unión.


In [ ]:
df_a.merge(df_b,how='left',left_on='id',right_on='id')

In [ ]:
df_a.merge(df_b,how='inner',left_on='id',right_on='id')

In [ ]:
merged= pm25_agg.merge( gbd, on=['country_name', 'year_id'], how='inner')
merged

### Herramientas visuales

Para visualizar datos podemos usar **matplotlib** y **seaborn**, dos librerías muy comunes en análisis de datos. Con comandos como `df['col'].value_counts().plot(kind='bar')` generamos gráficos rápidos y personalizados. Luego usamos `plt.title()`, `plt.ylabel()` y `plt.show()` para ajustar y mostrar la gráfica.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pm25['country_name'].value_counts().plot(kind='bar', figsize=(10,5))
plt.title('Número de registros por país')
plt.ylabel('Conteo')
plt.show()

In [ ]:
sns.boxplot(data=pm25[pm25['country_name']=='Argentina'], x='value')
plt.title('Distribución de PM2.5')
plt.show()

In [ ]:
pm25['value'].hist(bins=5, figsize=(10, 5))
plt.title('Distribución de los niveles de PM2.5')
plt.xlabel('Niveles de PM2.5 (µg/m³)')
plt.ylabel('Frecuencia (Número de Registros)')
plt.show()

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨